<a href="https://colab.research.google.com/github/MelB18/EasyQuanten/blob/main/Dekohaerenz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Klassische Wahrscheinlichkeitsverteilung

In [3]:
import numpy as np

# 1. Wahrscheinklichteisdichteverteilungen definieren (Beispiel: 75% für 0 und 25% für 1)
p_0 = 0.75
p_1 = 1.0 - p_0

# 2. Simulation von 1000 Messungen (Zufallsauswahl basierend auf den Wahrscheinlichkeiten)
anzahl_messungen = 1000
ergebnisse = np.random.choice([0, 1], size=anzahl_messungen, p=[p_0, p_1])

# 3. Auswertung
counts_0 = np.sum(ergebnisse == 0)
counts_1 = np.sum(ergebnisse == 1)

print(f"Ergebnis nach {anzahl_messungen} Messungen")
print(f"0: {counts_0} mal gemessen ({counts_0/anzahl_messungen*100:.1f}%)")
print(f"1: {counts_1} mal gemessen ({counts_1/anzahl_messungen*100:.1f}%)")


Ergebnis nach 1000 Messungen
0: 735 mal gemessen (73.5%)
1: 265 mal gemessen (26.5%)


2 verschränkte Qubits

In [5]:
%pip install qiskit
%pip install qiskit_aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 99.2 MB/s eta 0:00:00


In [8]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# 1. Quantenschaltkreis mit 2 Qubits und 2 klassischen Bits erstellen
qc = QuantumCircuit(2, 2)

# 2. Verschränkung erzeugen (Bell-Zustand |Φ+>)
qc.h(0)          # Setzt das erste Qubit in Superposition (|0> + |1>)
qc.cx(0, 1)      # CNOT-Gatter verschränkt Qubit 0 (Control) mit Qubit 1 (Target)

# 3. Messung beider Qubits
qc.measure([0, 1], [0, 1])

# 4. Simulation auf einem idealen Quantencomputer-Simulator
simulator = AerSimulator()
result = simulator.run(qc, shots=1000).result()
counts = result.get_counts(qc)

# 5. Ergebnis ausgeben
print("Messergebnisse nach 1000 Durchläufen:")
print(f"Zustand |00>: {counts.get('00', 0)} mal")
print(f"Zustand |11>: {counts.get('11', 0)} mal")
print(f"Fehlkombinationen (|01> oder |10>): {counts.get('01', 0) + counts.get('10', 0)} mal")


Messergebnisse nach 1000 Durchläufen:
Zustand |00>: 480 mal
Zustand |11>: 520 mal
Fehlkombinationen (|01> oder |10>): 0 mal


Wenn beide Qubits in der Z-Basis gemessen werden, sind die Ergebnisse perfekt korreliert (00 oder 11).

Wenn beide Qubits in der X-Basis gemessen werden, sind die Ergebnisse ebenfalls perfekt korreliert (++ oder --).

Wenn wir jedoch Mischmessungen durchführen (Qubit 1 in Z-Basis, Qubit 2 in X-Basis), bricht die Korrelation komplett zusammen! Jede der vier Kombinationen (0+, 0-, 1+, 1-) tritt dann mit einer Wahrscheinlichkeit von 25 % auf.

In [11]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def simuliere_basis(messung_q0_in_x, messung_q1_in_x):
    qc = QuantumCircuit(2, 2)

    # 1. Verschränkung erzeugen (|Φ+>)
    qc.h(0)
    qc.cx(0, 1)

    # 2. Basisänderung vor der Messung (Zustandsrotation)
    if messung_q0_in_x:
      qc.h(0) # Rotiert Qubit 0 von X-Basis nach Z-Basis
    if messung_q1_in_x:
      qc.h(1) # Rotiert Qubit 1 von X-Basis nach Z-Basis

    # 3. Messung
    qc.measure([0, 1], [0, 1])

    # Simulation ausführen
    simulator = AerSimulator()
    return simulator.run(qc, shots=1000).result().get_counts()

# Szenario A: Beide Qubits in der X-Basis messen
print("Szenario A (Beide X-Basis):", simuliere_basis(messung_q0_in_x=True, messung_q1_in_x=True))
# Ergebnis zeigt perfekte Korrelation (Bitstrings '00' steht für '++', '11' für '--')

# Szenario B: Gekreuzte Messung (Q0 in Z-Basis, Q1 in X-Basis)
print("Szenario B (Gekreuzte Basis):", simuliere_basis(messung_q0_in_x=False, messung_q1_in_x=True))
# Ergebnis zeigt Gleichverteilung (Verschränkungseffekt für diese Kombination unsichtbar)


Szenario A (Beide X-Basis): {'11': 494, '00': 506}
Szenario B (Gekreuzte Basis): {'01': 259, '11': 248, '10': 246, '00': 247}


Simulation eines partiellen Kollapses

In [ ]:
import numpy as np

# 1. Startzustand: Perfekte Superposition (|0> + |1>) / sqrt(2)
alpha = 1.0 / np.sqrt(2)
beta = 1.0 / np.sqrt(2)

# Messstärke (Stärke des Eingriffs pro Schritt)
# Ein kleiner Wert bedeutet eine sehr sanfte, schwache Messung
p = 0.15

print("Ausgangszustand:  P(|0>) = 50.0%, P(|1>) = 50.0%")
print("-" * 50)

# Wir führen 5 aufeinanderfolgende schwache Messungen durch
for schritt in range(1, 6):
    # Aktuelle Wahrscheinlichkeiten berechnen
    prob_0 = np.abs(alpha)**2
    prob_1 = np.abs(beta)**2

    # Wahrscheinlichkeit für das Messergebnis des schwachen Detektors
    # (Das Messgerät schaut nur mit reduzierter Effizienz hin)
    detektor_prob_0 = prob_0 + (1 - p) * prob_1

    # Der Zufall entscheidet, was der Detektor in diesem Schritt sieht
    if np.random.rand() < detektor_prob_0:
        # Detektor liefert Signal '0' -> Zustand wird leicht Richtung |0> verändert
        alpha = alpha
        beta = beta * np.sqrt(1 - p)
        ergebnis = "0 (Tendenz zu |0>)"
    else:
        # Detektor liefert Signal '1' -> Zustand wird leicht Richtung |1> verändert
        alpha = alpha * np.sqrt(1 - p)
        beta = beta
        ergebnis = "1 (Tendenz zu |1>)"

    # Zustand neu normieren (da die Amplituden geschrumpft sind)
    norm = np.sqrt(np.abs(alpha)**2 + np.abs(beta)**2)
    alpha /= norm
    beta /= norm

    # Aktuelle Wahrscheinlichkeiten nach dem teilweisen Kollaps
    neues_p0 = np.abs(alpha)**2
    neues_p1 = np.abs(beta)**2

    print(f"Schritt {schritt}: Detektor las {ergebnis}")
    print(f"          Neuer Zustand: P(|0>) = {neues_p0*100:.1f}%, P(|1>) = {neues_p1*100:.1f}%")